# Clustering de Patrones de Consumo Energético
## K-Means Clustering

Este notebook implementa clustering para identificar patrones de consumo energético.

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Añadir directorio src al path
sys.path.append('../src')

from preprocessing import load_data, clean_data
from clustering import (
    train_clustering_model,
    predict_cluster,
    calculate_inertia,
    visualize_clusters_pca,
    get_cluster_characteristics,
    save_model
)

from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Carga y Preparación de Datos

In [ ]:
# Cargar y limpiar datos
df = load_data('../data/raw/energy_consumption.csv')
df_clean = clean_data(df)

print(f'Registros originales: {len(df)}')
print(f'Registros después de limpieza: {len(df_clean)}')

df_clean.head()

## 2. Selección de Features para Clustering

In [ ]:
# Seleccionar features relevantes para clustering
feature_cols = ['pies_cuadrados', 'temperatura_aire', 'hora_del_dia', 'dia_de_la_semana', 'consumo_energia']
X = df_clean[feature_cols].copy()

# Normalizar features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print('Features seleccionadas:')
print(feature_cols)
print('\nEstadísticas después de escalado:')
display(X_scaled_df.describe())

## 3. Método del Codo para Determinar K Óptimo

In [ ]:
# Calcular inercia para diferentes valores de K
k_values, inertias = calculate_inertia(X_scaled_df, max_k=10)

# Graficar método del codo
plt.figure(figsize=(10, 6))
plt.plot(k_values, inertias, marker='o', linewidth=2, markersize=8)
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia')
plt.title('Método del Codo para Determinar K Óptimo')
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.show()

print('Valores de inercia por K:')
for k, inertia in zip(k_values, inertias):
    print(f'K={k}: {inertia:.2f}')

## 4. Entrenamiento de Modelos K-Means

In [ ]:
# Probar con diferentes valores de K
k_values_to_test = [3, 4, 5]
models = {}

for k in k_values_to_test:
    print(f'\nEntrenando modelo con K={k}...')
    model = train_clustering_model(X_scaled_df, n_clusters=k)
    models[k] = model
    print(f'Inercia: {model.inertia_:.2f}')
    print(f'Iteraciones: {model.n_iter_}')

## 5. Visualización de Clusters (K=3)

In [ ]:
# Usar K=3 como modelo final
k_final = 3
final_model = models[k_final]
labels = predict_cluster(final_model, X_scaled_df)

# Visualizar con PCA
fig = visualize_clusters_pca(X_scaled_df, labels, title=f'Clusters de Consumo Energético (K={k_final})')
plt.show()

# Contar elementos por cluster
unique, counts = np.unique(labels, return_counts=True)
print('\nDistribución de clusters:')
for cluster, count in zip(unique, counts):
    print(f'Cluster {cluster}: {count} registros ({count/len(labels)*100:.1f}%)')

## 6. Caracterización de Clusters

In [ ]:
# Obtener características de cada cluster
cluster_stats = get_cluster_characteristics(df_clean.iloc[:len(labels)], labels, feature_cols)

print('Estadísticas por Cluster:')
display(cluster_stats)

# Visualizar características promedio por cluster
means = cluster_stats.xs('mean', level=1, axis=1)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(feature_cols):
    means[col].plot(kind='bar', ax=axes[idx], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    axes[idx].set_title(f'{col} - Promedio por Cluster')
    axes[idx].set_xlabel('Cluster')
    axes[idx].grid(True, alpha=0.3)

# Ocultar el sexto subplot si no se usa
if len(feature_cols) < 6:
    axes[5].set_visible(False)

plt.tight_layout()
plt.show()

## 7. Interpretación de Clusters

In [ ]:
# Análisis de características principales de cada cluster
df_with_clusters = df_clean.iloc[:len(labels)].copy()
df_with_clusters['cluster'] = labels

print('=== INTERPRETACIÓN DE CLUSTERS ===\n')

for cluster in sorted(df_with_clusters['cluster'].unique()):
    cluster_data = df_with_clusters[df_with_clusters['cluster'] == cluster]
    print(f'\n--- Cluster {cluster} ---')
    print(f'Tamaño: {len(cluster_data)} registros')
    print(f'Consumo promedio: {cluster_data["consumo_energia"].mean():.2f} kWh')
    print(f'Temperatura promedio: {cluster_data["temperatura_aire"].mean():.2f}°C')
    print(f'Hora más común: {cluster_data["hora_del_dia"].mode().values[0]}')
    print(f'Área promedio: {cluster_data["pies_cuadrados"].mean():.0f} pies²')

## 8. Guardar Modelo

In [ ]:
# Guardar modelo y scaler
import joblib

model_path = '../models/clustering_model.pkl'
scaler_path = '../models/clustering_scaler.pkl'

# Crear directorio si no existe
Path('../models').mkdir(exist_ok=True)

# Guardar
save_model(final_model, model_path)
joblib.dump(scaler, scaler_path)

print(f'Modelo guardado en: {model_path}')
print(f'Scaler guardado en: {scaler_path}')
print(f'Número de clusters: {k_final}')

## 9. Conclusiones

### Resultados del Clustering:

1. Se identificaron **3 clusters** principales de patrones de consumo energético.
2. Cada cluster representa un perfil de consumo diferente basado en hora, temperatura y área.
3. El modelo puede utilizarse para segmentar el consumo y aplicar estrategias específicas.

### Aplicaciones:

- **Cluster 0**: Perfil de consumo bajo - Horarios de baja actividad
- **Cluster 1**: Perfil de consumo medio - Horarios normales de operación
- **Cluster 2**: Perfil de consumo alto - Horarios pico o condiciones extremas

### Recomendaciones:

1. Usar los clusters para personalizar estrategias de eficiencia energética.
2. Monitorear cambios de cluster en tiempo real para detectar anomalías.
3. Aplicar tarifas diferenciadas según el cluster de consumo.